# 02 — RAG Pipeline

Build a complete Retrieval-Augmented Generation pipeline: load local documents, chunk, embed, store in a vector database, and query with context grounding.

## Setup

Initialize the LLM and embedding model.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embed_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

print("Models ready!")

## Load Dummy Knowledge Documents

We use fictional internal documents about Acme Corp that the LLM has never seen in training.

In [ ]:
from pathlib import Path

doc_dir = Path("../docs/dummy-knowledge")
documents = {}
for filepath in sorted(doc_dir.glob("*.md")):
    content = filepath.read_text(encoding="utf-8")
    documents[filepath.name] = content
    print(f"Loaded: {filepath.name} ({len(content)} chars)")

## Step 11 — Document Chunking

Large documents don't fit in the LLM's context window efficiently. We split them into smaller overlapping chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n## ", "\n### ", "\n", " ", ""],
)

all_chunks = []
all_metadatas = []
for filename, content in documents.items():
    chunks = splitter.split_text(content)
    for c in chunks:
        all_chunks.append(c)
        all_metadatas.append({"source": filename})

print(f"Total chunks created: {len(all_chunks)}")
for i, chunk in enumerate(all_chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:200] + "...")

## Step 12 — Vector Storage (Qdrant In-Memory)

Store the chunks as dense vector embeddings in Qdrant's in-memory mode — no server needed.

In [ ]:
from langchain_qdrant import QdrantVectorStore

db = QdrantVectorStore.from_texts(
    all_chunks,
    embed_model,
    metadatas=all_metadatas,
    location=":memory:",
)

print(f"Vector store created with {len(all_chunks)} vectors")

## Step 13 — Assemble the RAG Pipeline

Wire the retriever into a `RetrievalQA` chain. The flow is: **Query → Retrieve relevant chunks → Inject into prompt → Generate answer**.

In [ ]:
from langchain.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 2}),
)

print("RAG pipeline ready!")

### Test: Query without RAG vs. with RAG

First, ask the bare LLM a question it can't possibly know. Then ask the RAG pipeline.

In [ ]:
# Without RAG — LLM alone
bare_response = llm.invoke(
    "What is the rate limit for the free tier of the Quantum Weather API?"
)
print("=== BARE LLM (no context) ===")
print(bare_response.content)

In [ ]:
# With RAG — grounded in our documents
rag_response = rag_chain.invoke(
    {"query": "What is the rate limit for the free tier of the Quantum Weather API?"}
)
print("=== RAG PIPELINE ===")
print(rag_response["result"])

### Show the Retrieved Context

See which chunks were injected into the prompt to ground the answer.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke("What is the rate limit for the free tier of the Quantum Weather API?")

print("Retrieved chunks:")
for i, doc in enumerate(docs):
    print(f"\n--- Doc {i+1} (source: {doc.metadata['source']}) ---")
    print(doc.page_content[:300])

### More Test Queries

Try asking about Acme Corp's remote work policy, the Project Aurora postmortem, or the Quantum Weather API SDKs.

In [ ]:
test_queries = [
    "How many paid leave days do Acme Corp employees get?",
    "What was the root cause of the Project Aurora outage?",
    "What Python SDK is available for the Quantum Weather API?",
    "What should I do if I get a 429 error?",
]

for q in test_queries:
    print(f"\n=== Q: {q} ===")
    result = rag_chain.invoke({"query": q})
    print(result["result"][:400])